# ICA Tutorial 1: Data Analysis with Python
Teaching notebook. Run the cells **top to bottom, one at a time** while you teach. Section numbers match the Updated Tutorial document.

**Before you start:** upload these 4 files using the Files panel (folder icon on the left): `students.csv`, `salary.csv`, `exercise.csv`, `ica_messy_student_dataset.csv`.

In [ ]:
import pandas as pd
import numpy as np

print(pd.__version__)   # Part 6 needs Pandas 2.0 or newer

---
## Part 3: Pandas and DataFrames
### 3.3 Loading data

In [ ]:
df = pd.read_csv("students.csv")
df

### 3.4 Creating a DataFrame from a dictionary
Keys become column names, lists become the columns. All lists must be the same length.

In [ ]:
data = {
    "Name": ["Ali", "Sara", "Ahmed"],
    "Age": [19, 20, 18],
    "Marks": [85, 72, 91]
}

df_small = pd.DataFrame(data)
df_small

A second way: a list of dictionaries, where each dictionary is one row.

In [ ]:
rows = [
    {"Name": "Ali", "Age": 19, "Marks": 85},
    {"Name": "Sara", "Age": 20, "Marks": 72},
    {"Name": "Ahmed", "Age": 18, "Marks": 91}
]

pd.DataFrame(rows)

### 3.5 The inspection commands
Run one at a time and say what each shows. Back on the demo dataset `df`.

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.sample(5)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.describe()

---
## Part 4: Finding problems
**Activity first:** look at the table and list anything that looks wrong.

In [ ]:
df

### Problem 1: Missing values

In [ ]:
df.isnull()          # True/False for every cell

In [ ]:
df.isnull().sum()    # count of missing per column

### Problem 2: Duplicate records

In [ ]:
df.duplicated()          # True for repeated rows

In [ ]:
df.duplicated().sum()    # how many repeats

In [ ]:
df[df.duplicated()]      # show the repeated rows

### Problem 3: Wrong data type

In [ ]:
salary = pd.read_csv("salary.csv")
salary.dtypes

### Problem 4: Inconsistent values

In [ ]:
df["Department"].unique()

In [ ]:
df["Department"].value_counts()

### Problem 5: Unusual values and outliers

In [ ]:
df.describe()      # look at the max of Age

In [ ]:
df[df["Age"] > 100]   # investigate first, do not guess

---
## Part 5: Data cleaning with Pandas
**Board order:** duplicates, then text fixes, then wrong types, then impossible values, then missing values.

Start by loading the file again into `messy_file`. This is also your **reset button**.

In [ ]:
messy_file = pd.read_csv("students.csv")

### Method 1: clean a copy
**5.1 Make a working copy**

In [ ]:
clean_file = messy_file.copy()

**5.2 Removing duplicates** (8 rows to 7)

In [ ]:
clean_file = clean_file.drop_duplicates()
clean_file.shape

**5.3 Fixing inconsistent values**

In [ ]:
clean_file["Department"] = clean_file["Department"].str.strip()
clean_file["Department"] = clean_file["Department"].str.upper()

In [ ]:
clean_file["Department"].value_counts()   # CS 3, BBA 2, MBA 2

**5.4 Fixing data types** (this cell reloads salary first, so you can re-run it)

In [ ]:
salary = pd.read_csv("salary.csv")
salary["Salary"] = salary["Salary"].str.replace("Rs. ", "")
salary["Salary"] = salary["Salary"].astype(int)
salary.dtypes

**5.5 Handling the outlier** (age 250 is impossible, so set it to missing)

In [ ]:
clean_file.loc[clean_file["Age"] > 100, "Age"] = np.nan
clean_file

**5.6 Missing values**

Option A: *show* the data without missing rows. This does **not** change `clean_file`.

In [ ]:
clean_file.dropna()

Option B: fill with the median. Expected: median age 20, median marks 81.5.

In [ ]:
clean_file["Age"] = clean_file["Age"].fillna(clean_file["Age"].median())
clean_file["Marks"] = clean_file["Marks"].fillna(clean_file["Marks"].median())
clean_file

### Method 2: clean the original directly (short variation)
Reload first, so `messy_file` is messy again. Every change is permanent for this DataFrame.

In [ ]:
messy_file = pd.read_csv("students.csv")

messy_file = messy_file.drop_duplicates()
messy_file["Department"] = messy_file["Department"].str.strip().str.upper()
messy_file.loc[messy_file["Age"] > 100, "Age"] = np.nan
messy_file["Age"] = messy_file["Age"].fillna(messy_file["Age"].median())
messy_file["Marks"] = messy_file["Marks"].fillna(messy_file["Marks"].median())

In [ ]:
messy_file.isnull().sum()
messy_file.duplicated().sum()
messy_file.shape        # expected (7, 4)

Warning: never write `messy_file = messy_file.drop_duplicates(inplace=True)`. It turns your data into `None`.

---
## Part 6: Cleaning messier, real-world data
Placeholders, fees with currency labels, mixed date formats. If you re-run cells here, start again from the copy cell (6.1).
### 6.1 Load and inspect

In [ ]:
messy_students = pd.read_csv("ica_messy_student_dataset.csv")

messy_students

In [ ]:
messy_students.info()

In [ ]:
clean = messy_students.copy()

### 6.2 Turn text placeholders into real missing values

In [ ]:
clean = clean.replace(["not available", "not recorded", ""], pd.NA)
clean.isnull().sum()    # Age 1, Registration_Date 1, Course_Fee 1, Marks 1

### 6.3 Convert numeric-looking text into numbers

In [ ]:
clean["Course_Fee"] = (
    clean["Course_Fee"]
    .astype("string")
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
clean["Course_Fee"] = pd.to_numeric(clean["Course_Fee"], errors="coerce")
clean["Course_Fee"]

### 6.4 Clean the Age column

In [ ]:
clean["Age"] = pd.to_numeric(clean["Age"], errors="coerce")

clean[clean["Age"].isna()]     # missing or could not be converted (Sara, Zara)

In [ ]:
clean[clean["Age"] > 100]      # plausibility check (Usman, 250)

### 6.5 Standardise the text categories

In [ ]:
clean["Department"] = clean["Department"].astype("string").str.strip().str.upper()
clean["Department"].unique()    # CS, BBA, ECONOMICS

### 6.6 Convert dates (skip if short on time)

In [ ]:
clean["Registration_Date"] = pd.to_datetime(
    clean["Registration_Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)
clean["Registration_Date"]

To *display* the dates as DD/MM/YYYY (this gives text, so keep the datetime column for calculations):

In [ ]:
clean["Registration_Date"].dt.strftime("%d/%m/%Y")

### 6.7 Finish the job with the same steps as before

In [ ]:
clean = clean.drop_duplicates()
clean.loc[clean["Age"] > 100, "Age"] = np.nan

clean["Age"] = clean["Age"].fillna(clean["Age"].median())
clean["Marks"] = clean["Marks"].fillna(clean["Marks"].median())
clean["Course_Fee"] = clean["Course_Fee"].fillna(clean["Course_Fee"].median())

clean

In [ ]:
clean.isnull().sum()   # only Registration_Date has 1 missing (left on purpose)
clean.shape            # expected (7, 7)

---
## Part 7: Check your cleaned data
Did the cleaning actually work? (Using the Method 1 result `clean_file`.)

In [ ]:
clean_file.head()

In [ ]:
clean_file.info()

In [ ]:
clean_file.isnull().sum()

In [ ]:
clean_file.duplicated().sum()

---
## Part 8: Student exercise
Give students this dataset. Let them try first. (Also available as `exercise.csv`.)

In [ ]:
data = {
    "Name": ["Ali", "Sara", "Ahmed", "Fatima", "Sara", "Usman"],
    "Age": [19, None, 18, 19, None, 250],
    "Department": ["CS", "BBA", "cs", "BBA", "BBA", "CS"],
    "Marks": [85, 90, None, 88, 90, 65]
}

df = pd.DataFrame(data)
df

In [ ]:
# Or load it from the file instead:
# df = pd.read_csv("exercise.csv")

---
## ANSWER KEY (for you only, do not show students)

In [ ]:
# Inspection
df.head()                    # 1. first rows
df.shape                     # 2. (6, 4)
df.isnull().sum()            # 3. Age: 2 missing, Marks: 1 missing
df.duplicated().sum()        # 4. 1 (the second Sara)
df["Department"].unique()    # 5. 'CS', 'BBA', 'cs' -> cs should be CS
df.describe()                # 6. Age max is 250, which is impossible

In [ ]:
# Cleaning
cleaned = df.copy()                                                    # 7
cleaned = cleaned.drop_duplicates()                                    # 8. now 5 rows
cleaned["Department"] = cleaned["Department"].str.strip().str.upper()  # 10
cleaned.loc[cleaned["Age"] > 100, "Age"] = np.nan                      # 250 treated as missing
cleaned["Age"] = cleaned["Age"].fillna(cleaned["Age"].median())        # 9. fills with 19
cleaned["Marks"] = cleaned["Marks"].fillna(cleaned["Marks"].median())  # 9. fills with 86.5

# Check
cleaned.info()
cleaned.isnull().sum()       # 11. all zeros
cleaned.duplicated().sum()   # 0